# T31 — Does the model depend less on the background? (lung vs background perturbation)

Inference only; nothing is trained. This is the run that decides whether the paper may say
"shortcut suppression" or only "moves evidence into the lungs".

For the baseline (A0) and the selected arm of each backbone at seeds 42, 123 and 2026, plus
DenseNet121 A3_multiply at seed 42, this notebook:
1. checks the checkpoint reproduces its committed test accuracy;
2. runs the **dependence test**: blur and different-class swap applied to the lungs, to the background,
   and to a background region with the same pixel count as the lungs; per image it records the
   total-variation change in the predicted distribution (`dP`) and `LRG = dP_lung - dP_background`;
3. runs the older stress tests (background zero / shuffle / noise, and lung/background occlusion);
4. compares the selected arm with A0 image by image, with bootstrap 95% intervals;
5. fills the DenseNet121 gap: Grad-CAM EIL for A3_multiply and A5_cbam, then the same significance
   tests the other backbones have.

**Expected run time on one T4:** about 2–3 hours for the full grid of 25 checkpoints
(roughly 2 min per EfficientNet-B0, 3 min per ResNet50, 4 min per DenseNet121 and 8 min per ViT-Base
checkpoint, plus about 10 min of setup, caching and the DenseNet Grad-CAM section).
Set `DRY_RUN = True` in the first cell for a 2-checkpoint trial of about 15 minutes.

**Before running (Kaggle sidebar):**
1. Session options → Accelerator → **GPU T4** (one is enough). Internet → **On**.
2. Add input → the primary dataset `tawsifurrahman/covid19-radiography-database`.
3. Add input → every dataset that holds the trained `.pt` checkpoints. The notebook searches all of
   `/kaggle/input` for `.pt` files and reads the backbone, arm and seed stored inside each one, so folder
   names do not matter. The seed-42 checkpoints are already in `uom230334h/t28-checkpoints`,
   `uom230334h/t28-vitbase-checkpoints` and `uom230334h/t28-efficientnet-checkpoints`. The seed 123 and
   2026 checkpoints (A0 and the selected arm of each backbone) still have to be uploaded as a dataset.
4. Run All. The inventory cell prints which checkpoints were found and which core-grid ones are missing;
   the run continues with whatever is present.

**Output:** `/kaggle/working/T31_dependence.zip`. The files to read first are
`dependence_paired_summary.csv` (selected minus A0, with intervals) and `dependence_verdict.csv`.

In [ ]:
import os, sys, json, time, math, hashlib, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/Ravindu-Pathirana/Chest-X-ray-Disease-Detection.git"
REPO_BRANCH = "t26-vit-kaggle-results"   # holds the ViT-Base results; falls back to main if it has been merged and deleted

# Everything below is found automatically under /kaggle/input. The environment variables only exist
# so the notebook can also be dry-run off Kaggle.
DATA_ROOT = Path(os.environ.get("TRUST_DATA_ROOT", "/kaggle/input"))   # searched for COVID-19_Radiography_Dataset
CKPT_ROOT = Path(os.environ.get("TRUST_CKPT_ROOT", "/kaggle/input"))   # searched recursively for *.pt
WORK      = Path(os.environ.get("TRUST_WORK", "/kaggle/working"))
REPO      = Path(os.environ.get("TRUST_REPO", str(WORK / "Chest-X-ray-Disease-Detection")))
OUT       = WORK / "T31_dependence"

BATCH_SIZE  = 64
NUM_WORKERS = int(os.environ.get("TRUST_WORKERS", "4"))
DRY_RUN     = os.environ.get("TRUST_DRY_RUN", "0") == "1"   # set to True for a 2-checkpoint trial run

RUN_DENSENET_GAP   = True    # Grad-CAM EIL for DenseNet121 A3_multiply and A5_cbam (section 5)
INCLUDE_ALL_SEED42 = False   # True: also run the dependence test on every seed-42 arm (adds about 1.5 h)
N_BOOT = 2000

OUT.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT, "| dry run:", DRY_RUN)

In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    if subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_BRANCH]).returncode != 0:
        print(f"branch {REPO_BRANCH} not found -> staying on the default branch")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo commit:", COMMIT)

for module, package in [("timm", "timm"), ("pytorch_grad_cam", "grad-cam")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if str(WORK).startswith("/kaggle") and DEVICE.type != "cuda":
    raise RuntimeError("No GPU. In the Kaggle sidebar set Session options -> Accelerator -> GPU T4, then Run All again.")

In [ ]:
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
SEEDS = [42, 123, 2026]
ARTIFACT = {"densenet121": "T18_lung_attention", "resnet50": "T23_resnet50_lung_attention",
            "efficientnet_b0": "T25_efficientnet_b0_lung_attention",
            "vit_base_patch16_224": "T26_vit_base_lung_attention"}
BASE_ARM = {"densenet121": "A0_vanilla", "resnet50": "A0_vanilla", "efficientnet_b0": "A0",
            "vit_base_patch16_224": "A0_vanilla"}
SELECTED_ARM = {"densenet121": "A2_full", "resnet50": "A3_multiply", "efficientnet_b0": "A3",
                "vit_base_patch16_224": "A3_multiply"}
SKIP_DIRS = {"images", "masks", ".venv", ".git", "__pycache__", "stage_2_train_images", "stage_2_test_images"}


def find_data_dir(root):
    for d, subdirs, _ in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        p = Path(d)
        if (p / "COVID" / "images").is_dir() and (p / "Viral Pneumonia" / "masks").is_dir():
            return p
    raise FileNotFoundError(f"COVID-19_Radiography_Dataset (with images/ and masks/) not found under {root}. "
                            "Attach the dataset tawsifurrahman/covid19-radiography-database.")


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def sha256_12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


def reference_dir(backbone, arm, seed):
    """Folder in the repo that holds this run's committed results."""
    base = REPO / "artifacts" / ARTIFACT[backbone]
    if backbone == "densenet121" and arm == "A2_bg":
        return base / "bg_suppression_followup" / "runs" / "A2_bg"
    if seed == 42:
        return base / "runs" / arm
    if backbone == "efficientnet_b0":
        return base / "runs_multiseed" / f"seed_{seed}" / "runs" / arm
    return base / "runs_multiseed" / f"{arm}_seed{seed}"


def discover_checkpoints(root):
    rows = []
    for d, subdirs, files in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        for name in files:
            path = Path(d) / name
            if not name.endswith(".pt") or "smoke" in str(path).lower():
                continue
            ck = torch_load(path)
            if not (isinstance(ck, dict) and "model_state_dict" in ck and "config" in ck):
                print("skipped (not a project checkpoint):", path)
                continue
            seed = int(ck.get("seed", 42))
            arm = str(ck.get("arm", path.stem))
            if arm.endswith(f"_seed{seed}"):
                arm = arm[: -len(f"_seed{seed}")]
            model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
            backbone = model_cfg["name"]
            if backbone not in ARTIFACT:
                print("skipped (unknown backbone):", path)
                continue
            rows.append({
                "backbone": backbone, "arm": arm, "seed": seed,
                "use_attention": module_cfg.get("use_attention"), "attention": module_cfg.get("attention", "lung"),
                "gate_mode": module_cfg.get("gate_mode"), "lambda_att": module_cfg.get("lambda_att"),
                "lambda_bg": module_cfg.get("lambda_bg"),
                "size_mb": round(path.stat().st_size / 2**20, 1), "path": str(path),
            })
            del ck
    inv = pd.DataFrame(rows)
    if inv.empty:
        raise FileNotFoundError(f"no .pt checkpoints found under {root}. Attach the checkpoint datasets.")
    before = len(inv)
    inv = inv.sort_values(["backbone", "seed", "arm", "path"]).drop_duplicates(["backbone", "arm", "seed"])
    if len(inv) < before:
        print(f"{before - len(inv)} duplicate checkpoint files ignored (same backbone/arm/seed)")
    inv["role"] = [
        "baseline" if a == BASE_ARM[b] else "selected" if a == SELECTED_ARM[b] else "other"
        for b, a in zip(inv.backbone, inv.arm)]
    inv["run"] = inv.backbone + "/" + inv.arm + "/seed_" + inv.seed.astype(str)
    return inv.reset_index(drop=True)


DATA_DIR = find_data_dir(DATA_ROOT)
print("dataset:", DATA_DIR)
INVENTORY = discover_checkpoints(CKPT_ROOT)
print(f"{len(INVENTORY)} checkpoints found")
print(INVENTORY.groupby(["backbone", "seed"]).size().unstack(fill_value=0))

# What the core grid needs: baseline + selected arm at three seeds for every backbone (24 checkpoints).
have = set(zip(INVENTORY.backbone, INVENTORY.arm, INVENTORY.seed))
missing_core = [(b, a, s) for b in ARTIFACT for a in (BASE_ARM[b], SELECTED_ARM[b]) for s in SEEDS
                if (b, a, s) not in have]
print("core grid checkpoints missing:", len(missing_core))
for m in missing_core:
    print("   MISSING", m)

In [ ]:
from src.datasets import build_dataloaders
from src.modules import build_model


def load_model(row):
    """Rebuild the model from the settings stored inside the checkpoint, then load the weights strictly."""
    ck = torch_load(row["path"])
    model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
    if list(ck.get("class_names", CLASS_NAMES)) != CLASS_NAMES:
        raise ValueError(f"class order differs in {row['path']}")
    model = build_model(
        num_classes=model_cfg["num_classes"], use_attention=module_cfg["use_attention"],
        reduction=module_cfg.get("reduction", 8), gate_mode=module_cfg.get("gate_mode", "residual"),
        backbone_name=model_cfg["name"], pretrained=False,
        attention=module_cfg.get("attention", "lung"), drop_rate=model_cfg.get("drop_rate", 0.0))
    model.load_state_dict(ck["model_state_dict"], strict=True)
    return model.to(DEVICE).eval()


def free(model):
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


@torch.no_grad()
def predict_logits(model, X):
    """fp32 logits for a cached image tensor (no AMP: fp16 inputs flip about 2% of predictions)."""
    out = []
    for i in range(0, len(X), BATCH_SIZE):
        y = model(X[i:i + BATCH_SIZE].to(DEVICE))
        out.append((y[0] if isinstance(y, tuple) else y).float().cpu())
    return torch.cat(out)


def image_key(path):
    return "/".join(str(path).replace("\\", "/").split("/")[-3:])


def cache_split(loader, with_masks):
    xs, ys, ms = [], [], []
    for batch in loader:
        xs.append(batch[0]); ys.append(batch[1])
        if with_masks:
            ms.append(batch[2])
    return torch.cat(xs), torch.cat(ys), (torch.cat(ms) if with_masks else None)


def verify_against_committed(row, probs, labels, keys):
    """Compare this session's test predictions with the results committed in the repo."""
    ref = reference_dir(row["backbone"], row["arm"], row["seed"])
    pred = probs.argmax(1).numpy()
    acc = float((pred == labels.numpy()).mean())
    out = {"test_accuracy": acc, "committed_accuracy": None, "pred_agreement": None,
           "max_abs_prob_diff": None, "verified": "no committed reference"}
    js, cs = ref / "phase2_finetune_test_results.json", ref / "per_image_predictions.csv"
    if js.exists():
        cm = np.array(json.loads(js.read_text())["confusion_matrix"])
        out["committed_accuracy"] = float(np.trace(cm) / cm.sum())
        out["verified"] = "ok" if abs(acc - out["committed_accuracy"]) <= 1e-4 else "ACCURACY MISMATCH"
    if cs.exists():
        df = pd.read_csv(cs)
        df["key"] = df["image_path"].map(image_key)
        df = df.set_index("key").loc[keys]
        ref_pred = df["pred_label"].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
        ref_prob = df[[f"prob_{k}" for k in range(4)]].to_numpy()
        out["pred_agreement"] = float((ref_pred == pred).mean())
        out["max_abs_prob_diff"] = float(np.abs(ref_prob - probs.numpy()).max())
    return out


t0 = time.time()
_, val_loader, test_loader, class_names, _, datasets = build_dataloaders(
    DATA_DIR, img_size=224, batch_size=BATCH_SIZE, seed=42, num_workers=NUM_WORKERS,
    split_manifest_path="artifacts/splits/split_manifest_v1.csv")
assert list(class_names) == CLASS_NAMES, class_names
assert len(datasets["val"]) == 3175 and len(datasets["test"]) == 3175, "fixed split must be 3,175 val / 3,175 test"
TEST_DS = datasets["test"]
TEST_KEYS = [image_key(TEST_DS.base_dataset.samples[TEST_DS.indices[i]][0]) for i in range(len(TEST_DS))]
assert len(set(TEST_KEYS)) == len(TEST_KEYS)

X_test, y_test, M_test = cache_split(test_loader, with_masks=True)
if M_test.ndim == 3:
    M_test = M_test.unsqueeze(1)
print(f"cached test images {tuple(X_test.shape)} and masks {tuple(M_test.shape)} in {time.time() - t0:.0f}s")
print("mean lung fraction of the image:", float((M_test > 0.5).float().mean()))


class CachedTestDataset(torch.utils.data.Dataset):
    """The fixed test split served from RAM. Same tensors and order as the real test loader,
    and it exposes base_dataset/indices so the repo's evaluation code can read image paths."""

    def __init__(self, real, X, Y, M):
        self.base_dataset, self.indices = real.base_dataset, real.indices
        self.X, self.Y, self.M = X, Y, M

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], int(self.Y[i]), self.M[i]


class CachedLoader:
    def __init__(self, dataset, batch_size):
        self.dataset, self.batch_size = dataset, batch_size
        self.sampler = torch.utils.data.SequentialSampler(dataset)

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        d = self.dataset
        for i in range(0, len(d), self.batch_size):
            yield d.X[i:i + self.batch_size], d.Y[i:i + self.batch_size], d.M[i:i + self.batch_size]


CACHED_LOADER = CachedLoader(CachedTestDataset(TEST_DS, X_test, y_test, M_test), BATCH_SIZE)

## 1. Checks before any result is produced

The perturbation code's own unit tests, then a direct check on real images that background modes leave
every lung pixel untouched and lung modes leave every background pixel untouched. A picture of each
perturbation is saved to `perturbation_examples.png` — look at it once before trusting the numbers.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_counterfactual.py", "tests/test_xai_dependence.py"],
                   capture_output=True, text=True)
print((r.stdout or r.stderr).strip().splitlines()[-1])

from src.modules import perturb_background, perturb_lung
import matplotlib
import matplotlib.pyplot as plt

SWAP_PAIRS = pd.read_csv(REPO / "artifacts/explainable_ai/swap_pairs_seed42.csv")
assert set(SWAP_PAIRS.image_path) == set(TEST_KEYS), "frozen donor pairs do not match the test split"
key_to_pos = {k: i for i, k in enumerate(TEST_KEYS)}
donor_of = dict(zip(SWAP_PAIRS.image_path, SWAP_PAIRS.donor_image_path))

show = [int(np.where(y_test.numpy() == c)[0][0]) for c in range(4)]      # first test image of each class
x, m = X_test[show], M_test[show]
donors = torch.stack([X_test[key_to_pos[donor_of[TEST_KEYS[i]]]] for i in show])
lung = (m > 0.5).expand_as(x)
torch.manual_seed(42)
variants = {"original": x}
for mode in ["blur", "swap", "zero", "shuffle", "noise"]:
    v = perturb_background(x, m, mode=mode, donor_images=donors if mode == "swap" else None)
    assert torch.equal(v[lung], x[lung]), f"background {mode} changed lung pixels"
    variants[f"background {mode}"] = v
for mode in ["blur", "swap", "zero"]:
    v = perturb_lung(x, m, mode=mode, donor_images=donors if mode == "swap" else None)
    assert torch.equal(v[~lung], x[~lung]), f"lung {mode} changed background pixels"
    variants[f"lung {mode}"] = v
print("region checks passed: background modes keep lungs identical, lung modes keep background identical")

mean, std = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1), torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
fig, axes = plt.subplots(len(show), len(variants), figsize=(2.1 * len(variants), 2.2 * len(show)))
for c, (name, v) in enumerate(variants.items()):
    for r_ in range(len(show)):
        axes[r_, c].imshow((v[r_] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r_, c].axis("off")
        if r_ == 0:
            axes[r_, c].set_title(name, fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "perturbation_examples.png", dpi=110)
plt.show()

## 2. Run the grid

Each checkpoint is verified first. A checkpoint whose accuracy does not match the committed value is
skipped and reported, so a wrong file cannot produce numbers.

In [ ]:
from src.modules import evaluate_counterfactual_robustness, evaluate_region_occlusion, evaluate_xai_dependence

grid = INVENTORY[INVENTORY.role.isin(["baseline", "selected"])]
extra = INVENTORY[(INVENTORY.backbone == "densenet121") & (INVENTORY.arm == "A3_multiply") & (INVENTORY.seed == 42)]
if INCLUDE_ALL_SEED42:
    extra = pd.concat([extra, INVENTORY[INVENTORY.seed == 42]])
grid = pd.concat([grid, extra]).drop_duplicates(["backbone", "arm", "seed"]).sort_values(["backbone", "seed", "arm"])
if DRY_RUN:
    grid = grid.head(2)
print(len(grid), "checkpoints in this run")

verification, failed = [], []
for n, row in enumerate(grid.to_dict("records"), 1):
    t1 = time.time()
    out_dir = OUT / row["backbone"] / row["arm"] / f"seed_{row['seed']}"
    if (out_dir / "done.json").exists():
        verification.append(json.loads((out_dir / "done.json").read_text()))
        print(f"[{n}/{len(grid)}] {row['run']}: already done, skipped")
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    model = load_model(row)
    check = verify_against_committed(row, F.softmax(predict_logits(model, X_test), dim=1), y_test, TEST_KEYS)
    info = {"backbone": row["backbone"], "arm": row["arm"], "seed": row["seed"], "role": row["role"], **check,
            "checkpoint": row["path"], "checkpoint_sha256_12": sha256_12(row["path"]), "repo_commit": COMMIT}
    if check["verified"] == "ACCURACY MISMATCH":
        failed.append(info)
        print(f"[{n}/{len(grid)}] {row['run']}: SKIPPED, accuracy {check['test_accuracy']:.4f} "
              f"!= committed {check['committed_accuracy']:.4f}")
        free(model)
        continue

    per_image, dep = evaluate_xai_dependence(model, CACHED_LOADER, CLASS_NAMES, SWAP_PAIRS, DEVICE,
                                            arm=row["arm"], seed=row["seed"], pairing_seed=42)
    per_image.drop(columns=["original_probabilities", "perturbed_probabilities"]).to_csv(
        out_dir / "dependence_per_image.csv.gz", index=False)
    dep.to_csv(out_dir / "dependence_summary.csv", index=False)

    evaluate_counterfactual_robustness(
        model, CACHED_LOADER, DEVICE, seed=row["seed"], arm_name=row["arm"],
        per_image_csv=out_dir / "counterfactual_per_image.csv").to_csv(out_dir / "counterfactual_summary.csv", index=False)
    evaluate_region_occlusion(
        model, CACHED_LOADER, DEVICE, arm_name=row["arm"],
        per_image_csv=out_dir / "occlusion_per_image.csv").to_csv(out_dir / "occlusion_summary.csv", index=False)
    free(model)

    info["minutes"] = round((time.time() - t1) / 60, 2)
    (out_dir / "done.json").write_text(json.dumps(info, indent=2))
    verification.append(info)
    lrg = dep.set_index("mode")["LRG"]
    print(f"[{n}/{len(grid)}] {row['run']}: acc {check['test_accuracy']:.4f} ({check['verified']})  "
          f"LRG blur {lrg['blur']:+.4f}  swap {lrg['swap']:+.4f}  ({info['minutes']} min)")

pd.DataFrame(verification + failed).to_csv(OUT / "checkpoint_verification.csv", index=False)
print("finished:", len(verification), "| skipped for accuracy mismatch:", len(failed))

## 3. Collect the per-checkpoint summaries

In [ ]:
def collect(filename):
    frames = []
    for v in verification:
        p = OUT / v["backbone"] / v["arm"] / f"seed_{v['seed']}" / filename
        if p.exists():
            df = pd.read_csv(p)
            for col in ("role", "seed", "backbone"):
                df[col] = v[col]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


dep_all, cf_all, occ_all = collect("dependence_summary.csv"), collect("counterfactual_summary.csv"), collect("occlusion_summary.csv")
dep_all.to_csv(OUT / "dependence_summary_all.csv", index=False)
cf_all.to_csv(OUT / "counterfactual_summary_all.csv", index=False)
occ_all.to_csv(OUT / "occlusion_summary_all.csv", index=False)

# The zero/shuffle/noise stress tests can push every image to one class. When that happens the flip
# rate only reflects how many images were not in that class to begin with, and says nothing about
# background reliance. top_class_share close to 1.0 flags that case.
sat = []
for v in verification:
    p = OUT / v["backbone"] / v["arm"] / f"seed_{v['seed']}" / "counterfactual_per_image.csv"
    if p.exists():
        d = pd.read_csv(p)
        for mode, g in d.groupby("mode"):
            share = g.perturbed_prediction.value_counts(normalize=True)
            sat.append({"backbone": v["backbone"], "arm": v["arm"], "seed": v["seed"], "mode": mode,
                        "top_class_after_perturbation": CLASS_NAMES[int(share.index[0])],
                        "top_class_share": float(share.iloc[0])})
if sat and len(cf_all):
    cf_all = cf_all.merge(pd.DataFrame(sat), on=["backbone", "arm", "seed", "mode"], how="left")
    cf_all.to_csv(OUT / "counterfactual_summary_all.csv", index=False)
    print(cf_all[["backbone", "arm", "seed", "mode", "pred_flip_rate", "top_class_after_perturbation",
                  "top_class_share"]].round(4).to_string(index=False))
    print()

pd.set_option("display.width", 220)
print(dep_all[["backbone", "arm", "seed", "mode", "mean_dP_lung", "mean_dP_background", "mean_dP_bg_area_matched",
               "LRG", "lung_flip_rate", "background_flip_rate"]].round(4).to_string(index=False))

# Harness check: the committed EfficientNet-B0 seed-42 stress-test numbers were produced by the same
# function on Kaggle. "zero" is deterministic and should match closely; shuffle/noise use random numbers.
ref_cf = REPO / "artifacts/T25_efficientnet_b0_lung_attention/counterfactual_summary.csv"
mine = cf_all[(cf_all.backbone == "efficientnet_b0") & (cf_all.seed == 42)] if len(cf_all) else cf_all
if ref_cf.exists() and len(mine):
    cmp_ = mine.merge(pd.read_csv(ref_cf), on=["arm", "mode"], suffixes=("_this_run", "_committed"))
    print("\nEfficientNet-B0 seed 42, this run vs committed:")
    print(cmp_[["arm", "mode", "pred_flip_rate_this_run", "pred_flip_rate_committed",
                "mean_stability_this_run", "mean_stability_committed"]].round(4).to_string(index=False))

## 4. Selected arm vs A0, paired by image

For each backbone, seed and mode the difference (selected − A0) is taken image by image, with a
percentile bootstrap 95% interval (2,000 resamples, seed 42).

- `LRG` up means the model relies relatively more on the lungs than on the background.
- `dP_background` down means the prediction moves less when the background is changed.
- `LRG_area_matched` repeats LRG against background pixels equal in number to the lung pixels (see the
  caution below).
- `donor_flip_rate` (swap only): how often the prediction flips to the class of the image whose
  background was pasted in. This is the most direct sign that the model reads the background.

**Decision rule, fixed before looking:** reduced background reliance is supported on a backbone only if
the LRG difference has an interval above zero in both blur and swap on every available seed.

**About the area-matched control:** it is reported but not used in the rule. As implemented in
`xai_dependence.py` it picks scattered background pixels, not one contiguous patch. In a trial run on
DenseNet121 that speckle pattern moved predictions about twice as much as perturbing the whole
background (dP 0.42 against 0.22 under blur), so it measures sensitivity to an unnatural texture
rather than to background area. Treat those columns with caution.

In [ ]:
def wide_per_image(v):
    df = pd.read_csv(OUT / v["backbone"] / v["arm"] / f"seed_{v['seed']}" / "dependence_per_image.csv.gz")
    w = df.pivot(index=["image_path", "mode"], columns="region", values="dP")
    w["LRG"] = w["lung"] - w["background"]
    w["LRG_area_matched"] = w["lung"] - w["background_area_matched"]
    bg = df[df.region == "background"].set_index(["image_path", "mode"])
    w["donor_flip"] = bg["flip_to_donor_class"].astype(float)
    w["bg_flip"] = bg["prediction_flipped"].astype(float)
    return w.rename(columns={"lung": "dP_lung", "background": "dP_background",
                             "background_area_matched": "dP_bg_area_matched"})


def boot_ci(diff, rng):
    idx = rng.integers(0, len(diff), size=(N_BOOT, len(diff)))
    means = diff[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


done = {(v["backbone"], v["arm"], v["seed"]): v for v in verification}
rows = []
for backbone in ARTIFACT:
    for seed in SEEDS:
        a0, sel = done.get((backbone, BASE_ARM[backbone], seed)), done.get((backbone, SELECTED_ARM[backbone], seed))
        if not (a0 and sel):
            continue
        base, new = wide_per_image(a0), wide_per_image(sel)
        assert base.index.equals(new.index)
        for mode in ["blur", "swap"]:
            b, s = base.xs(mode, level="mode"), new.xs(mode, level="mode")
            rec = {"backbone": backbone, "seed": seed, "mode": mode, "baseline_arm": a0["arm"],
                   "selected_arm": sel["arm"], "n_images": len(b)}
            rng = np.random.default_rng(42)
            for col in ["LRG", "LRG_area_matched", "dP_lung", "dP_background", "dP_bg_area_matched"]:
                d = (s[col] - b[col]).to_numpy()
                lo, hi = boot_ci(d, rng)
                rec.update({f"{col}_baseline": b[col].mean(), f"{col}_selected": s[col].mean(),
                            f"{col}_diff": d.mean(), f"{col}_ci_low": lo, f"{col}_ci_high": hi})
            rec.update({"bg_flip_rate_baseline": b["bg_flip"].mean(), "bg_flip_rate_selected": s["bg_flip"].mean(),
                        "donor_flip_rate_baseline": b["donor_flip"].mean(), "donor_flip_rate_selected": s["donor_flip"].mean()})
            rows.append(rec)
paired = pd.DataFrame(rows)
paired.to_csv(OUT / "dependence_paired_summary.csv", index=False)
if len(paired):
    print(paired[["backbone", "seed", "mode", "LRG_baseline", "LRG_selected", "LRG_diff", "LRG_ci_low", "LRG_ci_high",
                  "LRG_area_matched_diff", "dP_background_diff", "donor_flip_rate_baseline", "donor_flip_rate_selected"]]
          .round(4).to_string(index=False))

    def verdict(g):
        up = g.LRG_ci_low > 0
        down = (g.LRG_ci_high < 0)
        return pd.Series({
            "seeds": g.seed.nunique(), "comparisons": len(g), "LRG_higher_with_CI": int(up.sum()),
            "LRG_lower_with_CI": int(down.sum()),
            "verdict": "reduced background reliance SUPPORTED" if up.all()
                       else "selected arm relies MORE on background" if down.all() else "not shown / mixed"})

    verdicts = pd.DataFrame({b: verdict(g) for b, g in paired.groupby("backbone")}).T
    verdicts.to_csv(OUT / "dependence_verdict.csv")
    print()
    print(verdicts.to_string())

In [ ]:
# Does a higher Grad-CAM EIL go with lower background dependence? (per image, 1,000-image CAM subset)
from scipy.stats import spearmanr

rows = []
for v in verification:
    ref = reference_dir(v["backbone"], v["arm"], v["seed"]) / "per_image_predictions.csv"
    if not ref.exists():
        continue
    eil = pd.read_csv(ref)
    eil["image_path"] = eil["image_path"].map(image_key)
    eil = eil.dropna(subset=["eil_post"]).set_index("image_path")["eil_post"]
    w = wide_per_image(v)
    for mode in ["blur", "swap"]:
        dp = w.xs(mode, level="mode")["dP_background"].reindex(eil.index)
        rho, p = spearmanr(eil.to_numpy(), dp.to_numpy())
        rows.append({"backbone": v["backbone"], "arm": v["arm"], "seed": v["seed"], "mode": mode,
                     "n": len(eil), "spearman_eil_vs_dP_background": rho, "p_value": p})
corr = pd.DataFrame(rows)
corr.to_csv(OUT / "eil_vs_background_dependence.csv", index=False)
print("A clearly negative value would mean EIL is a usable stand-in for background dependence.")
print(corr.round(4).to_string(index=False))

## 5. DenseNet121 gap: Grad-CAM EIL for A3_multiply and A5_cbam

These two arms were trained but never Grad-CAM-scored, so the winner rule used on the other backbones
could not be applied to DenseNet121. A2_full is re-scored first as a check on the harness: its mean EIL
must come out at the committed 0.3735.

In [ ]:
from src.modules import build_per_image_predictions, run_significance_tests
from scipy.stats import binomtest

gap_dir = OUT / "densenet121_gap"
dn42 = INVENTORY[(INVENTORY.backbone == "densenet121") & (INVENTORY.seed == 42)].set_index("arm")
if RUN_DENSENET_GAP and {"A3_multiply", "A5_cbam", "A2_full"} <= set(dn42.index):
    positions = json.loads((REPO / "artifacts/T18_lung_attention/cam_subset_indices.json").read_text())["positions"]
    if DRY_RUN:
        positions = positions[:48]
    cam_subset = np.array(positions)
    new = {}
    for arm in ["A2_full", "A3_multiply", "A5_cbam"]:
        t1 = time.time()
        out_dir = gap_dir / arm
        out_dir.mkdir(parents=True, exist_ok=True)
        model = load_model(dn42.loc[arm].to_dict())
        df = build_per_image_predictions(model, TEST_DS, CLASS_NAMES, DEVICE, cam_subset=cam_subset, batch_size=32)
        free(model)
        df.to_csv(out_dir / "per_image_predictions.csv", index=False)
        metrics = {"cam_eil_post_mean": float(df.eil_post.mean()), "cam_eil_pre_mean": float(df.eil_pre.mean()),
                   "cam_subset_n": int(df.eil_post.notna().sum()), "attention_ilar": float(df.ilar.mean()),
                   "accuracy": float((df.true_label == df.pred_label).mean())}
        (out_dir / "attention_metrics.json").write_text(json.dumps(metrics, indent=2))
        new[arm] = df
        print(f"{arm}: EIL post {metrics['cam_eil_post_mean']:.4f}  pre {metrics['cam_eil_pre_mean']:.4f}  "
              f"acc {metrics['accuracy']:.4f}  ({time.time() - t1:.0f}s)")
    if not DRY_RUN:
        assert abs(new["A2_full"].eil_post.mean() - 0.3735) < 0.005, "harness check failed: A2_full EIL is not 0.3735"
        print("harness check passed: A2_full EIL matches the committed value")

    # Same paired tests as T23/T25/T26, over every DenseNet121 arm that now has per-image EIL.
    arms = {}
    for arm in ["A0_vanilla", "A1_gate_only", "A2_full", "A4_guidance_only"]:
        p = REPO / "artifacts/T18_lung_attention/runs" / arm / "per_image_predictions.csv"
        if p.exists():
            arms[arm] = pd.read_csv(p)
    arms.update({"A3_multiply": new["A3_multiply"], "A5_cbam": new["A5_cbam"]})
    for arm, df in arms.items():
        df = df.copy()
        df["image_path"] = df["image_path"].map(image_key)
        arms[arm] = {"per_image_df": df}
    sig = run_significance_tests(arms, "A0_vanilla", output_json=gap_dir / "significance_tests_vs_A0.json")
    table = pd.DataFrame(sig).T
    # One-sided version of the accuracy gate (only a significant DROP fails), as T25 uses.
    table["mcnemar_one_sided_arm_worse_p"] = [
        1.0 if (r.mcnemar_b + r.mcnemar_c) == 0 else
        float(binomtest(int(r.mcnemar_b), int(r.mcnemar_b + r.mcnemar_c), 0.5, alternative="greater").pvalue)
        for r in table.itertuples()]
    table["candidate_two_sided_rule"] = (table.mcnemar_p >= 0.05) & (table.wilcoxon_p < 0.05) & (table.mean_eil_post_gain > 0)
    table["candidate_one_sided_rule"] = (table.mcnemar_one_sided_arm_worse_p >= 0.05) & (table.wilcoxon_p < 0.05) & (table.mean_eil_post_gain > 0)
    table.to_csv(gap_dir / "T18_significance_table.csv")
    print(table[["mcnemar_b", "mcnemar_c", "mcnemar_p", "mcnemar_one_sided_arm_worse_p", "wilcoxon_p",
                 "mean_eil_post_gain", "candidate_two_sided_rule", "candidate_one_sided_rule"]].to_string())
    for rule in ["candidate_two_sided_rule", "candidate_one_sided_rule"]:
        cands = table[table[rule]]
        winner = cands.mean_eil_post_gain.astype(float).idxmax() if len(cands) else None
        print(f"DenseNet121 winner under {rule}: {winner}")
else:
    print("DenseNet121 gap section skipped (switched off, or the seed-42 A2/A3/A5 checkpoints are not attached)")

In [ ]:
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Download this file from the Output panel:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MB)")